In [280]:
import pandas as pd
import numpy as np
from plotly.express import histogram

from tqdm import tqdm

import plotly.express as px
import plotly.graph_objs as go

import Irina.utility_functions as uf

In [413]:
path = uf.PATH+"cosine_pipeline/"
start_year = 2000
end_year = 2023

# Country by Year
## Productivity (N of articles) and heterogeneity

In [19]:
df_country_stats = pd.read_csv(path+"df_country_stats_compare.csv")

In [112]:
years = [start_year, end_year]
df_country_stats_plot = (
    df_country_stats
    .merge(uf.df_country[["alpha-2", "name", "sub-region", "region"]],
           left_on="country",
           right_on="alpha-2",
           how="left")
    .rename(columns={"sub-region": "subregion"})
    .drop("alpha-2", axis=1)
    .assign(
        total_articles_norm = lambda df: df.groupby("country")["total_articles"].transform(lambda x: x / x.iloc[0]),
        total_articles_norm_unique = lambda df: df.groupby("country")["total_articles_unique"].transform(lambda x: x / x.iloc[0]),
        log_total_articles_norm = lambda df: np.log(df.total_articles_norm),
        log_total_articles_norm_unique = lambda df: np.log(df.total_articles_norm_unique),
        gini_pct_change=lambda df: df.groupby("country")["gini"].transform(lambda x: x - x.iloc[0]),
        gini_pct_change_unique=lambda df: df.groupby("country")["gini_unique"].transform(lambda x: x - x.iloc[0])
    )
)

In [113]:
(
    df_country_stats_plot
    .query(f"year == {end_year}")
    [["country", "name", "region", "subregion", "total_articles", "total_articles_unique",
      "total_articles_norm", "total_articles_norm_unique"]]
    .sort_values(by="total_articles_norm_unique", ascending=False)
).tail(30)

,country,name,region,subregion,total_articles,total_articles_unique,total_articles_norm,total_articles_norm_unique
5090,CU,Cuba,Americas,Latin America and the Caribbean,1968,1016.0,2.351254,1.795053
5203,PG,Papua New Guinea,Oceania,Melanesia,191,54.0,4.063830,1.741935
5114,GB,United Kingdom of Great Britain and Northern I...,Europe,Northern Europe,243971,119909.0,2.752039,1.715829
5176,MM,Myanmar,Asia,South-eastern Asia,588,256.0,3.127660,1.641026
5147,JP,Japan,Asia,Eastern Asia,145914,107762.0,1.854195,1.598985
5123,GP,Guadeloupe,Americas,Latin America and the Caribbean,197,51.0,3.581818,1.500000
5180,MQ,Martinique,Americas,Latin America and the Caribbean,118,33.0,4.370370,1.500000
5259,VE,"Venezuela, Bolivarian Republic of",Americas,Latin America and the Caribbean,2403,1436.0,1.937903,1.494277
5138,IM,Isle of Man,Europe,Northern Europe,12,4.0,1.333333,1.333333
5078,BZ,Belize,Americas,Latin America and the Caribbean,34,5.0,3.777778,1.250000


In [412]:
px.bar(
    df_country_stats_plot.query(f"year == {end_year}"),
    x="log_total_articles_norm_unique",
    y="name",
    height=1000,
    color="region"
)

In [411]:
px.bar(
    df_country_stats_plot.query(f"year == {end_year}"),
    x="gini_pct_change",
    y="name",
    height=1000,
    color="subregion",
    title=f"Gini Index (heterogeneity of subfield distribution) {end_year}-{start_year} (neg - more heterog)",
)

In [409]:
px.scatter(
    df_country_stats_plot.query(f"year == {end_year}"),
    x="log_total_articles_norm",
    y="log_total_articles_norm_unique",
    color="region",
    hover_data="name",
    title=f"log(N_articles_{end_year} / N_Articles {start_year})",
    labels={"log_total_articles_norm": "with collaborations", "log_total_articles_norm_unique": "without collaborations"},
)

In [408]:
px.scatter(
    df_country_stats_plot.query(f"year == {end_year}"),
    x="log_total_articles_norm_unique",
    y="gini_pct_change_unique",
    color="region",
    hover_data="name",
    title="Heterogeneity by subfields (neg gini - less heterogeneity)",
    labels={f"log_total_articles_norm_unique": f"log (N_articles_{end_year} / N_Articles {str(start_year)}) -- no collaborations",
            f"gini_pct_change_unique": f"(gini_{end_year} - gini_{start_year}) / gini_{start_year} -- heterogeneity",},
)

In [78]:
df_country_stats_plot.subregion.drop_duplicates()

0                         Western Asia
1                        Southern Asia
2      Latin America and the Caribbean
3                      Southern Europe
5                   Sub-Saharan Africa
7                            Polynesia
8                       Western Europe
9            Australia and New Zealand
17                      Eastern Europe
21                    Northern America
22                  South-eastern Asia
38                        Eastern Asia
48                     Northern Europe
51                     Northern Africa
59                           Melanesia
61                          Micronesia
99                        Central Asia
189                                NaN
Name: subregion, dtype: object

In [86]:
region = "Europe"
subregion = "Eastern Asia"
px.line(
    (
        df_country_stats_plot
        # .query(f"region == \"{region}\"")
        .query(f"subregion == \"{subregion}\"")
    ),
    x="year",
    y="gini_unique",
    color="country",
    hover_data="name",
    title="Gini Index (heterogeneity of subfield distribution)",
)

In [87]:
region = "Europe"
px.line(
    (
        df_country_stats_plot
        # .query(f"region == \"{region}\"")
        .query(f"subregion == \"{subregion}\"")
    ),
    x="year",
    y="total_articles_norm_unique",
    color="country",
    hover_data="name",
    title="Total unique articles (normalized by first-year value)",
)

# Distances by country yearly
## Compare Interest Metric within one country from 2000 to 2023

In [414]:
df_interest_metric_unique_list = []

for year in np.arange(start_year, end_year + 1):
    df_interest_metric_unique_list.append(
        pd.read_csv(path+f"df_interest_metric/unique_{str(year)}.csv").dropna(how="all").assign(year=year)
    )
df_interest_metric_unique = pd.concat(df_interest_metric_unique_list)

In [417]:
for country in tqdm(df_country_stats.country.drop_duplicates()):
    df_country_interest_metric = (
        df_interest_metric_unique.query(f"country == \"{country}\"").sort_values(by=["year"], ascending=True)
        .drop("country", axis=1)
        .set_index("year")
    )
    if df_country_interest_metric.empty:
        continue
    df_country_distance_yearly = uf.get_cosine_distances(df_country_interest_metric.fillna(0))

    idx = df_country_stats.query(f"country == \"{country}\" and year == {df_country_distance_yearly.index.to_list()}").index

    distance_yearly = np.concatenate([[0], np.diag(df_country_distance_yearly.values, k=1)])

    df_country_stats.loc[idx, "distance_yearly"] = distance_yearly
    df_country_stats.loc[idx, "distance_from_start"] = df_country_distance_yearly.values[0, :]

  0%|          | 0/226 [00:00<?, ?it/s]/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

  1%|          | 2/226 [00:00<00:11, 19.63it/s]/Users/irinavorobeva/Library

In [362]:
px.histogram(df_country_stats.query(f"year == {end_year}").distance_from_start)

In [297]:
country_list = ["FR", "US", "CN", "MD", "PA", "AL"]

px.line(
    df_country_stats.query(f"country == @country_list"),
    x="year",
    y=["distance_yearly", "distance_from_start"],
    title=f"Cosine distance yearly",
    color="country"
)

In [366]:
country = "CH"
years = [start_year, end_year]
df_plot = (
    df_interest_metric_unique
    .query(f"country == \"{country}\" and year == {years}")
    .set_index("year")
    .drop("country", axis=1)
    .T
)

px.line(df_plot,
        hover_name=[uf.id2subfield_topic[int(s)] for s in df_plot.index.to_list()],
        title=f"Interest metric, years compared: {uf.id2name_country[country]} ({country})",
        labels={"index": "subfield", "value": "interest metric"})

In [367]:
df_country_interest_metric = (
        df_interest_metric_unique.query(f"country == \"{country}\"").sort_values(by=["year"], ascending=True)
        .drop("country", axis=1)
        .set_index("year")
    )
df_country_distance_yearly = uf.get_cosine_distances(df_country_interest_metric.fillna(0))

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [368]:
x_labels = df_country_interest_metric.index.to_list()
y_labels = df_country_interest_metric.index.to_list()
uf.plotly_heatmap(
    df_country_distance_yearly,
    x_labels=x_labels,
    y_labels=y_labels,
    z_min=0, z_max=2,
    title=f"Difference in interest metric yearly: {uf.id2name_country[country]} ({country})",
)

In [369]:
x_labels = df_country_interest_metric.columns.to_list()
y_labels = df_country_interest_metric.index.to_list()
uf.plotly_heatmap(
    df_country_interest_metric,
    x_labels=x_labels,
    y_labels=y_labels,
    x_type="subfield",
    z_min=-3, z_max=3,
    title=f"Interest metric dynamics yearly: {uf.id2name_country[country]} ({country})",
)

In [370]:
uf.df_topics.query("subfield_id == 1409")

,topic_id,topic_name,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
2398,T12399,Factors Influencing Wine Tourism and Consumer ...,1409,"Tourism, Leisure and Hospitality Management",14,"Business, Management and Accounting",2,Social Sciences,Wine Tourism; Consumer Behavior; Wine Choice; ...,This cluster of papers focuses on the factors ...,https://en.wikipedia.org/wiki/Wine_tourism
2583,T12584,Challenges and Trends in Tourism Education and...,1409,"Tourism, Leisure and Hospitality Management",14,"Business, Management and Accounting",2,Social Sciences,Tourism Education; Hospitality Management; Car...,This cluster of papers explores the challenges...,https://en.wikipedia.org/wiki/Tourism_education


# Global interest dynamics
## In terms of subfield probability distributions

In [385]:
path = uf.PATH + "df_country_subfield_Floriana/"
name = "country_subfield_matrix_"
name_unique = "country_subfield_matrix_SingleCountry_"

df_country_subfield_list = []
df_country_subfield_unique_list = []

for year in np.arange(start_year, 2024):
    df_country_subfield_list.append(
        pd.read_csv(path+name+str(year)+".csv", index_col=0).assign(year=year).reset_index()
    )
    df_country_subfield_unique_list.append(
        pd.read_csv(path+name_unique+str(year)+".csv", index_col=0).assign(year=year).reset_index()
    )

df_country_subfield = pd.concat(df_country_subfield_list)
df_country_subfield_unique = pd.concat(df_country_subfield_unique_list)

In [387]:
df_world_subfield = (
    df_country_subfield.drop("country", axis=1).groupby("year").sum()
    .div(df_country_subfield.drop("country", axis=1).groupby("year").sum().sum(axis=1), axis=0)
)

df_world_subfield_unique = (
    df_country_subfield_unique.drop("country", axis=1).groupby("year").sum()
    .div(df_country_subfield_unique.drop("country", axis=1).groupby("year").sum().sum(axis=1), axis=0)
)

In [400]:
x_labels = df_world_subfield_unique.columns.to_list()
y_labels = df_world_subfield_unique.index.to_list()
uf.plotly_heatmap(
    df_world_subfield,
    x_labels=x_labels,
    y_labels=y_labels,
    x_type="subfield",
    colorscale="non-suymmetric",
    z_min=0, z_max=0.05,
    title=f"Global: Subfield Probability distribution yearly (with collaborations)",
)

In [401]:
x_labels = df_world_subfield_unique.columns.to_list()
y_labels = df_world_subfield_unique.index.to_list()
uf.plotly_heatmap(
    df_world_subfield_unique,
    x_labels=x_labels,
    y_labels=y_labels,
    x_type="subfield",
    colorscale="non-suymmetric",
    z_min=0, z_max=0.05,
    title=f"Global: Subfield Probability distribution yearly (without collaborations)",
)

In [402]:
x_labels = df_world_subfield_unique.columns.to_list()
y_labels = df_world_subfield_unique.index.to_list()
uf.plotly_heatmap(
    df_world_subfield.sub(df_world_subfield_unique),
    x_labels=x_labels,
    y_labels=y_labels,
    x_type="subfield",
    colorscale="symmetric",
    title=f"Global: Subfield Probability distribution difference (collaborations - no collaborations)",
)